In [0]:
df_silver_politicos_all_geral = spark.sql("""
    WITH df_silver_votacao_secao AS (
        SELECT * FROM workspace.drs_silver.votacao_secao_2024_go
    ),
    df_silver_socio_demografico AS (
        SELECT * FROM workspace.drs_silver.socio_demografico
    ),
    df_silver_covid_19 AS (
        SELECT * FROM workspace.drs_silver.covid_19
    ),
    df_silver_local_votacao AS (
        SELECT * FROM workspace.drs_silver.local_votacao
    ),
    df_silver_estabelecimentos_saude AS (
        SELECT * FROM workspace.drs_silver.estabelecimentos_saude_go
    )
    SELECT 
        a.*,
        e.* EXCEPT (ibge_city_code, state_abbreviation, city_name, neighborhood, ingestion_timestamp, source_file, silver_processed_timestamp, created_at, updated_at),
        c.* EXCEPT (city, city_ibge_code, state, ingestion_timestamp, source_file, silver_processed_timestamp, created_at, updated_at),
        d.* EXCEPT (neighborhood, section_code, zone_code, ingestion_timestamp, source_file, silver_processed_timestamp, created_at, updated_at),
        b.* EXCEPT (city_code, city_name, state_abbreviation, ingestion_timestamp, source_file, silver_processed_timestamp, created_at, updated_at)
    FROM df_silver_votacao_secao AS a
    LEFT JOIN df_silver_socio_demografico AS b
        ON b.city_name = a.city_name
    LEFT JOIN df_silver_covid_19 AS c
        ON c.city = a.city_name
    LEFT JOIN df_silver_local_votacao AS d
        ON d.section_code = a.section_number
        AND d.zone_code = a.zone_number
    LEFT JOIN df_silver_estabelecimentos_saude AS e
        ON e.neighborhood = d.neighborhood
""")
display(df_silver_politicos_all_geral)
df_silver_politicos_all_geral.printSchema

In [0]:
# salvando a tabela full geral

df_silver_politicos_all_geral.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("workspace.drs_gold.politicos_all_geral")

In [0]:
# Votação × Saúde por Município
# Cruzamento entre participação eleitoral e infraestrutura de saúde — indicador de correlação cívica/social.

df_gold_votacao_saude = spark.sql("""
    SELECT
        a.city_name,
        a.state_abbreviation,

        -- Votação
        COUNT(DISTINCT a.section_number)            AS total_secoes,
        COUNT(DISTINCT a.zone_number)               AS total_zonas,
        SUM(a.vote_count)                           AS total_votos,
        COUNT(DISTINCT d.polling_place)             AS total_locais_votacao,
        COUNT(DISTINCT d.neighborhood)              AS total_bairros_com_urna,

        -- Saúde
        COUNT(DISTINCT e.cnes_id)                   AS total_estabelecimentos_saude,
        COUNT(DISTINCT e.establishment_type)        AS total_tipos_estabelecimento,
        COUNT(DISTINCT e.neighborhood)              AS total_bairros_com_saude

    FROM workspace.drs_silver.votacao_secao_2024_go a
    LEFT JOIN workspace.drs_silver.local_votacao d
        ON d.section_code = a.section_number
        AND d.zone_code = a.zone_number
    LEFT JOIN workspace.drs_silver.estabelecimentos_saude_go e
        ON d.neighborhood = e.neighborhood
    GROUP BY a.city_name, a.state_abbreviation
""")

In [0]:
# Salvando a tabela gold by region
df_gold_votacao_saude.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.drs_gold.votacao_saude_por_municipio")

In [0]:
spark.table("workspace.drs_silver.socio_demografico").printSchema()


In [0]:
# COVID × Sociodemográfico por Município
# Indicador de impacto da pandemia cruzado com perfil populacional.

df_gold_covid_socio = spark.sql("""
    SELECT
        b.city_name,
        b.state_abbreviation,
        b.city_code,

        -- Sociodemográfico
        b.resident_population_2022                                          AS population,
        b.population_density_hab_km2_2022                                   AS population_density,
        b.gdp_per_capita_brl_2020                                           AS gdp_per_capita,
        b.hdi_2010                                                          AS hdi,
        b.infant_mortality_per_1000_births_2020                             AS infant_mortality,
        b.school_enrollment_6_14_years_2010                                 AS school_enrollment,
        b.territorial_area_km2_2022                                         AS territorial_area_km2,

        -- COVID
        MAX(c.last_available_confirmed)                                     AS total_confirmados,
        MAX(c.last_available_deaths)                                        AS total_obitos,
        MAX(c.last_available_date)                                          AS ultima_atualizacao_covid,
        COUNT(DISTINCT c.date)                                              AS dias_com_registro_covid,

        -- Indicadores derivados
        ROUND(
            MAX(c.last_available_deaths)
            / NULLIF(b.resident_population_2022, 0) * 100000, 2)           AS mortalidade_por_100k,

        ROUND(
            MAX(c.last_available_confirmed)
            / NULLIF(b.resident_population_2022, 0) * 100, 4)              AS taxa_incidencia_pct,

        ROUND(
            b.committed_expenses_brl_1000_2017
            / NULLIF(b.resident_population_2022, 0), 2)                    AS gasto_publico_per_capita,

        ROUND(
            b.realized_revenues_brl_1000_2017
            - b.committed_expenses_brl_1000_2017, 2)                       AS saldo_fiscal_brl_1000

    FROM workspace.drs_silver.socio_demografico b
    LEFT JOIN workspace.drs_silver.covid_19 c
        ON c.city = b.city_name
    GROUP BY
        b.city_name,
        b.state_abbreviation,
        b.city_code,
        b.resident_population_2022,
        b.population_density_hab_km2_2022,
        b.gdp_per_capita_brl_2020,
        b.hdi_2010,
        b.infant_mortality_per_1000_births_2020,
        b.school_enrollment_6_14_years_2010,
        b.territorial_area_km2_2022,
        b.committed_expenses_brl_1000_2017,
        b.realized_revenues_brl_1000_2017
""")
display(df_gold_covid_socio)


In [0]:
# Salvando a tabela covid_socio_por_municipio
df_gold_covid_socio.write \
    .format("delta")\
    .mode("overwrite") \
    .saveAsTable("workspace.drs_gold.covid_socio_por_municipio")

In [0]:
# Votação por Bairro (Padre Bernardo)
# Granularidade máxima — útil direto para o teu projeto do município.

df_gold_votacao_bairro = spark.sql("""
    SELECT
        a.city_name,
        d.neighborhood                              AS bairro,
        d.zone_code                                 AS zona,
        COUNT(DISTINCT a.section_number)            AS total_secoes,
        COUNT(DISTINCT d.polling_place)             AS total_locais,
        SUM(a.vote_count)                           AS total_votos,
        COUNT(DISTINCT a.ballot_item_name)          AS total_cargos_disputados

    FROM workspace.drs_silver.votacao_secao_2024_go a
    LEFT JOIN workspace.drs_silver.local_votacao d
        ON d.section_code = a.section_number
        AND d.zone_code = a.zone_number
    WHERE a.city_name = 'PADRE BERNARDO'
    GROUP BY a.city_name, d.neighborhood, d.zone_code
    ORDER BY total_votos DESC
""")

In [0]:
# salvando a tabela votacao_por_bairro_padre_bernardo

df_gold_votacao_bairro.write\
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.drs_gold.votacao_por_bairro_padre_bernardo")

In [0]:
# Saúde × Bairro — Cobertura Territorial
# Mostra quais bairros têm estabelecimentos de saúde e quais são desassistidos.

df_gold_cobertura_saude = spark.sql("""
    SELECT
        e.neighborhood                              AS bairro,
        e.establishment_type                        as tipo_estabelecimento,
        e.sus_affiliation                           AS afiliacao_sus,
        e.legal_name                                AS legal_name, 
        e.legal_nature                              AS natureza_juridica,
        COUNT(DISTINCT e.legal_name)                AS total_estabelecimentos,
        COUNT(DISTINCT e.establishment_type)        AS total_tipos_estabelecimentos,
        COUNT(DISTINCT d.polling_place)             AS total_locais_votacao_no_bairro,
        count(distinct e.sus_affiliation)           AS total_afiliacao_sus,

        -- Flag de cobertura
        CASE 
            WHEN COUNT(DISTINCT e.cnes_id) = 0 THEN 'Sem cobertura'
            WHEN COUNT(DISTINCT e.cnes_id) <= 2  THEN 'Cobertura baixa'
            WHEN COUNT(DISTINCT e.cnes_id) <= 5  THEN 'Cobertura média'
            ELSE 'Cobertura alta'
        END                                         AS nivel_cobertura_saude

    FROM workspace.drs_silver.local_votacao d
    LEFT JOIN workspace.drs_silver.estabelecimentos_saude_go e
        ON d.neighborhood = e.neighborhood
    GROUP BY e.neighborhood, e.establishment_type, e.sus_affiliation, e.legal_name, e.legal_nature
    ORDER BY total_estabelecimentos DESC
""")


In [0]:
# salvando a tabela cobertura_saude_por_bairro

df_gold_cobertura_saude.write\
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.drs_gold.cobertura_saude_por_bairro")

In [0]:
# Perfil Demográfico por Faixa Etária e Gênero
# Distribuição populacional pura — base para todos os outros indicadores.

df_gold_demografico = spark.sql("""
    SELECT
        f.age_group                                                         AS faixa_etaria,
        f.gender                                                            AS genero,

        -- População
        SUM(f.population)                                                   AS total_populacao,
        ROUND(AVG(f.age_years), 1)                                          AS idade_media,

        -- Participação relativa
        ROUND(
            SUM(f.population)
            / NULLIF(SUM(SUM(f.population)) OVER (), 0) * 100, 2)          AS pct_populacao_total

    FROM workspace.drs_silver.faixa_hetaria_padre_bernardo_22 f
    GROUP BY f.age_group, f.gender
    ORDER BY f.age_group, f.gender
""")

display(df_gold_demografico)

In [0]:
df_gold_demografico.write\
    .format("delta")\
    .mode("overwrite") \
    .saveAsTable("workspace.drs_gold.perfil_demografico_padre_bernardo")

In [0]:
# COVID × Faixa Etária (Vulnerabilidade Demográfica
# Cruza o peso demográfico de cada faixa com os dados de COVID do município — mostra quais grupos populacionais vivem num contexto de maior exposição.

df_gold_covid_faixa = spark.sql("""
    SELECT
        f.age_group                                                         AS faixa_etaria,
        f.gender                                                            AS genero,
        SUM(f.population)                                                   AS populacao_faixa,

        -- COVID do município
        MAX(c.last_available_confirmed)                                     AS total_confirmados_municipio,
        MAX(c.last_available_deaths)                                        AS total_obitos_municipio,

        -- Indicadores de vulnerabilidade relativa
        ROUND(
            SUM(f.population)
            / NULLIF(SUM(SUM(f.population)) OVER (), 0) * 100, 2)          AS pct_populacao_total,

        ROUND(
            MAX(c.last_available_deaths)
            / NULLIF(SUM(SUM(f.population)) OVER (), 0) * 100000, 2)       AS mortalidade_covid_por_100k,

        ROUND(
            MAX(c.last_available_confirmed)
            / NULLIF(SUM(SUM(f.population)) OVER (), 0) * 100, 4)          AS taxa_incidencia_pct

    FROM workspace.drs_silver.faixa_hetaria_padre_bernardo_22 f
    LEFT JOIN workspace.drs_silver.covid_19 c
        ON c.city = 'PADRE BERNARDO'
    GROUP BY f.age_group, f.gender
    ORDER BY f.age_group
""")

display(df_gold_covid_faixa)


In [0]:
# Salvando a tabela covid_por_faixa_etaria_padre_bernardo

df_gold_covid_faixa.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.drs_gold.covid_por_faixa_etaria_padre_bernardo")

In [0]:
# Votação × Faixa Etária (Potencial Eleitoral por Grupo)
# Cruza a população por faixa com os votos registrados — estima o peso eleitoral de cada grupo demográfico.

df_gold_eleitoral_faixa = spark.sql("""
    SELECT
        f.age_group                                                         AS faixa_etaria,
        f.gender                                                            AS genero,
        SUM(f.population)                                                   AS populacao_faixa,

        -- Votos do município
        SUM(v.vote_count)                                                   AS total_votos_municipio,
        COUNT(DISTINCT v.section_number)                                    AS total_secoes,
        COUNT(DISTINCT v.zone_number)                                       AS total_zonas,

        -- Indicador: peso demográfico vs participação eleitoral
        ROUND(
            SUM(f.population)
            / NULLIF(SUM(SUM(f.population)) OVER (), 0) * 100, 2)          AS pct_populacao,

        ROUND(
            SUM(v.vote_count)
            / NULLIF(SUM(f.population), 0) * 100, 2)                       AS votos_por_habitante_faixa

    FROM workspace.drs_silver.faixa_hetaria_padre_bernardo_22 f
    LEFT JOIN workspace.drs_silver.votacao_secao_2024_go v
        ON v.city_name = 'PADRE BERNARDO'
    GROUP BY f.age_group, f.gender
    ORDER BY populacao_faixa DESC
""")

display(df_gold_eleitoral_faixa)

In [0]:
# Salvando a tabela eleitoral_por_faixa_etaria_padre_bernardo
df_gold_eleitoral_faixa.write\
    .format("delta")\
    .mode("overwrite") \
    .saveAsTable("workspace.drs_gold.eleitoral_por_faixa_etaria_padre_bernardo")